### TAO remote client - Object Detection

Transfer learning is the process of transferring learned features from one application to another. It is a commonly used training technique where you use a model trained on one task and re-train to use it on a different task. Train Adapt Optimize (TAO) Toolkit  is a simple and easy-to-use Python based AI toolkit for taking purpose-built AI models and customizing them with users' own data.

![image](https://d29g4g2dyqv443.cloudfront.net/sites/default/files/akamai/TAO/tlt-tao-toolkit-bring-your-own-model-diagram.png)

### Sample prediction for an Object Detection model
<img align="center" src="../example_images/sample_object_detection.jpg" width="960">

### The workflow in a nutshell

- Pulling datasets from cloud
- Running dataset convert
- Getting a PTM from NGC
- Model Actions
    - Train (Normal/AutoML)
    - Evaluate
    - Prune, retrain
    - Export
    - Tao-Deploy
    - Inference on TAO, TRT
    - Delete experiments/dataset

### Table of contents

1. [Install TAO remote client ](#head-1)
1. [FIXME's](#head-2)
1. [Login](#head-3)
1. [Create a cloud workspace](#head-2)
1. [Set dataset formats](#head-4)
1. [Create and pull train dataset](#head-5)
1. [Create and pull val dataset](#head-6)
1. [Create and pull test dataset](#head-7)
1. [List the created datasets](#head-8)
1. [Train Dataset convert action](#head-9) (for specific models)
1. [Val dataset convert action](#head-10) (for specific models)
1. [Create experiment (via create-job)](#head-11)
1. [Assign train, eval datasets](#head-12)
1. [List experiments](#head-13)
1. [Assign PTM](#head-14)
1. [Set AutoML related configurations](#head-15)
1. [Train](#head-16)
1. [View hyperparameters that are enabled by default](#head-16.1)
1. [Evaluate](#head-17)
1. [Optimize: Prune, retrain and evaluate](#head-18)
1. [Export](#head-19)
1. [TRT Engine generation using TAO-Deploy](#head-20)
1. [TAO inference](#head-21)
1. [TRT inference](#head-22)
1. [Delete experiment](#head-23)
1. [Delete dataset](#head-24)

### Requirements
Please find the server requirements [here](https://docs.nvidia.com/tao/tao-toolkit/text/tao_toolkit_api/api_setup.html#)

### Install TAO remote client <a class="anchor" id="head-1"></a>

In [ ]:
%%bash
# SKIP if already installed.
pip3 install nvidia-tao-client

In [ ]:
%%bash
tao --version

### Bash-based notebook

This notebook uses **Bash** (e.g. `%%bash` cells). State between cells is kept in `.tao_nb_state` in this directory. Ensure `tao login` has been run and `jq` is installed if you parse JSON output. When piping command output to `jq`, use **--output json** so the CLI emits JSON (default is text).


In [ ]:
%%bash
touch .tao_nb_state
source .tao_nb_state
which jq >/dev/null 2>&1 || echo 'Install jq for JSON parsing (e.g. brew install jq)'


In [ ]:
%%bash
source .tao_nb_state
echo "State: MODEL_NAME=${MODEL_NAME:-not set} WORKSPACE_ID=${WORKSPACE_ID:-not set}"


### To see the dataset folder structure required for the models supported in this notebook, visit the notebooks under dataset_prepare like for [this notebook](../dataset_prepare/object_detection.ipynb)

### FIXME's <a class="anchor" id="head-2"></a>

1. Assign a model_name in FIXME 1
1. (Optional) Enable AutoML if needed in FIXME 2
1. (Optional) To use AutoML: run get-automl-defaults (e.g. --base-experiment-id ID --action train --output @automl_defaults.json) then create-job with --automl-settings @automl_defaults.json
1. Assign a workdir in FIXME 4 for log file download
1. Assign the ip_address and port_number in FIXME 5 ([info](https://docs.nvidia.com/tao/tao-toolkit/text/tao_toolkit_api/api_rest_api.html))
1. Assign the ngc_key variable in FIXME 6
1. Assign the ngc_org_name variable in FIXME 7
1. Set cloud storage details in FIXME 8
1. Assign path of datasets relative to the bucket in FIXME 9
1. Database backup/restore archive filename in FIXME 10

#### Choose a object detection model

In [ ]:
%%bash
source .tao_nb_state
export MODEL_NAME="${TAO_MODEL_NAME:-dino}"
echo "export MODEL_NAME=$MODEL_NAME" >> .tao_nb_state
echo "MODEL_NAME=$MODEL_NAME"


#### Toggle AutoML params
[AutoML documentation](https://docs.nvidia.com/tao/tao-toolkit/text/automl/automl.html#getting-started)

In [ ]:
%%bash
source .tao_nb_state
export AUTOML_ENABLED="${TAO_AUTOML_ENABLED:-false}"
export AUTOML_ALGORITHM="${TAO_AUTOML_ALGORITHM:-bayesian}"
echo "export AUTOML_ENABLED=$AUTOML_ENABLED" >> .tao_nb_state
echo "export AUTOML_ALGORITHM=$AUTOML_ALGORITHM" >> .tao_nb_state
echo "AUTOML_ENABLED=$AUTOML_ENABLED"


### Login <a class="anchor" id="head-3"></a>

### Common Functions used across the notebook

#### Function to load login details from saved config

#### Set API service's host information

In [ ]:
%%bash
source .tao_nb_state
# Optional: export TAO_BASE_URL=... TAO_ORG=... TAO_TOKEN=...


#### Set NGC Personal key for authentication and NGC org to access API services

In [ ]:
%%bash
source .tao_nb_state
export NGC_KEY="${NGC_KEY:-your_ngc_key}"
echo "export NGC_KEY=$NGC_KEY" >> .tao_nb_state


In [ ]:
%%bash
source .tao_nb_state
export NGC_ORG="${NGC_ORG:-nvstaging}"
echo "export NGC_ORG=$NGC_ORG" >> .tao_nb_state
echo "NGC_ORG=$NGC_ORG"


In [ ]:
%%bash
source .tao_nb_state
tao login --ngc-org-name "${NGC_ORG:-nvstaging}" --ngc-key "${NGC_KEY:-your_ngc_key}" --enable-telemetry


### Get GPU details <a class="anchor" id="head-2"></a>

 Use --backend-type lepton --platform-id <key> (or --backend-type slurm --partition <name>) when you run create-job. Keys from get-gpu-types can be used as the platform ID.

In [ ]:
%%bash
source .tao_nb_state
# Optional: get GPU types for Slurm/Lepton backends
# tao get-gpu-types --output json | jq .


### Create cloud workspace
This workspace will be the place where your datasets reside and your results of TAO API jobs will be pushed to.

If you want to have different workspaces for dataset and experiment, duplocate the workspace creation part and adjust the metadata accordingly.

In [ ]:
%%bash
source .tao_nb_state
# FIXME 7/8: Set workspace cloud details (env or edit below)
export TAO_WORKSPACE_NAME="${TAO_WORKSPACE_NAME:-AWS workspace info}"
export TAO_WORKSPACE_CLOUD_TYPE="${TAO_WORKSPACE_CLOUD_TYPE:-aws}"
export TAO_WORKSPACE_CLOUD_REGION="${TAO_WORKSPACE_CLOUD_REGION:-us-west-1}"
export TAO_WORKSPACE_CLOUD_BUCKET_NAME="${TAO_WORKSPACE_CLOUD_BUCKET_NAME:-bucket_name}"
export TAO_WORKSPACE_CLOUD_ACCESS_KEY="${TAO_WORKSPACE_CLOUD_ACCESS_KEY:-access_key}"
export TAO_WORKSPACE_CLOUD_SECRET_KEY="${TAO_WORKSPACE_CLOUD_SECRET_KEY:-secret_key}"
echo "export TAO_WORKSPACE_NAME=$TAO_WORKSPACE_NAME" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_TYPE=$TAO_WORKSPACE_CLOUD_TYPE" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_REGION=$TAO_WORKSPACE_CLOUD_REGION" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_BUCKET_NAME=$TAO_WORKSPACE_CLOUD_BUCKET_NAME" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_ACCESS_KEY=$TAO_WORKSPACE_CLOUD_ACCESS_KEY" >> .tao_nb_state
echo "export TAO_WORKSPACE_CLOUD_SECRET_KEY=$TAO_WORKSPACE_CLOUD_SECRET_KEY" >> .tao_nb_state


In [ ]:
%%bash
source .tao_nb_state
# Create workspace (AWS example; use create-workspace-azure etc. for other clouds)
WORKSPACE_RESPONSE=$(tao $MODEL_NAME create-workspace-aws --name "AWS Workspace" \
  --cloud-region "$TAO_WORKSPACE_CLOUD_REGION" \
  --cloud-bucket-name "$TAO_WORKSPACE_CLOUD_BUCKET_NAME" \
  --access-key "$TAO_WORKSPACE_CLOUD_ACCESS_KEY" --secret-key "$TAO_WORKSPACE_CLOUD_SECRET_KEY" --output json)
WORKSPACE_ID=$(echo "$WORKSPACE_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$WORKSPACE_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "WORKSPACE_ID=$WORKSPACE_ID"
echo "export WORKSPACE_ID=$WORKSPACE_ID" >> .tao_nb_state

#### Set dataset URI (URI within cloud storage)

URIs should point to your dataset location in cloud storage, e.g. `aws://bucket-name/path/to/dataset`.

In [ ]:
%%bash
source .tao_nb_state
# FIXME 8: Paths relative to cloud bucket
export TRAIN_DATASET_URI="${TAO_WORKSPACE_CLOUD_TYPE}://${TAO_WORKSPACE_CLOUD_BUCKET_NAME}/data/tao_od_synthetic_subset_train_no_convert"
export EVAL_DATASET_URI="${TAO_WORKSPACE_CLOUD_TYPE}://${TAO_WORKSPACE_CLOUD_BUCKET_NAME}/data/tao_od_synthetic_subset_val_no_convert"
echo "export TRAIN_DATASET_URI=$TRAIN_DATASET_URI" >> .tao_nb_state
echo "export EVAL_DATASET_URI=$EVAL_DATASET_URI" >> .tao_nb_state
echo "TRAIN_DATASET_URI=$TRAIN_DATASET_URI"
echo "EVAL_DATASET_URI=$EVAL_DATASET_URI"

export NUM_CLASSES=4
echo "export NUM_CLASSES=$NUM_CLASSES" >> .tao_nb_state
echo "NUM_CLASSES=$NUM_CLASSES"

In [ ]:
%%bash
source .tao_nb_state
# Optional: Restore workspace from backup
# tao $MODEL_NAME restore-workspace --workspace-id $WORKSPACE_ID --backup_file_name mongodump.tar.gz


In [ ]:
%%bash
source .tao_nb_state
# Set dataset type/format (edit as needed for your model)
export DS_TYPE="${DS_TYPE:-object_detection}"
if [ "$MODEL_NAME" = "grounding_dino" ]; then
  export DS_FORMAT="${DS_FORMAT:-odvg}"
else
  export DS_FORMAT="${DS_FORMAT:-coco}"
fi
echo "export DS_TYPE=$DS_TYPE" >> .tao_nb_state
echo "export DS_FORMAT=$DS_FORMAT" >> .tao_nb_state

### Set common params across all jobs <a class="anchor" id="head-10"></a>

In [ ]:
%%bash
source .tao_nb_state
export ENCODE_KEY="${ENCODE_KEY:-tlt_encode}"
export HF_TOKEN="${HF_TOKEN:-}"
echo "export ENCODE_KEY=$ENCODE_KEY" >> .tao_nb_state
echo "export HF_TOKEN=$HF_TOKEN" >> .tao_nb_state

### Assign PTM <a class="anchor" id="head-14"></a>

Search for PTM on NGC for the Object Detection model chosen

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME list-base-experiments --filter-param network_arch=$MODEL_NAME --output json | jq -r '.[] | "\(.name)\t\(.id)\t\(.network_arch)"'


In [ ]:
%%bash
source .tao_nb_state
# Select PTM by matching ngc_path suffix (like Python pretrained_map). Override PTM_NGC_PATH_SUFFIX or set SELECTED_PTM_ID to change

# Set PTM based on model
if [ "$MODEL_NAME" = "deformable_detr" ]; then
  PTM_NGC_PATH_SUFFIX="pretrained_deformable_detr_nvimagenet:resnet50"
elif [ "$MODEL_NAME" = "dino" ]; then
  PTM_NGC_PATH_SUFFIX="pretrained_dino_nvimagenet:resnet50"
elif [ "$MODEL_NAME" = "grounding_dino" ]; then
  PTM_NGC_PATH_SUFFIX="grounding_dino:grounding_dino_swin_tiny_commercial_trainable_v1.0"
elif [ "$MODEL_NAME" = "rtdetr" ]; then
  PTM_NGC_PATH_SUFFIX="rtdetr_2d_warehouse:trainable_efficientvit_l2_v1.0"
else
  PTM_NGC_PATH_SUFFIX="${PTM_NGC_PATH_SUFFIX:-pretrained_deformable_detr_nvimagenet:resnet50}"
fi
echo "export PTM_NGC_PATH_SUFFIX=$PTM_NGC_PATH_SUFFIX" >> .tao_nb_state
echo "PTM_NGC_PATH_SUFFIX=$PTM_NGC_PATH_SUFFIX"

if [ -z "$SELECTED_PTM_ID" ]; then
  JSON=$(tao $MODEL_NAME list-base-experiments --filter-param network_arch=$MODEL_NAME --output json)
  if [ -n "$PTM_NGC_PATH_SUFFIX" ]; then
    export SELECTED_PTM_ID=$(echo "$JSON" | jq -r --arg s "$PTM_NGC_PATH_SUFFIX" '.[] | select(.ngc_path != null and (.ngc_path | endswith($s))) | .id' | head -1)
  fi
  if [ -z "$SELECTED_PTM_ID" ]; then
    export SELECTED_PTM_ID=$(echo "$JSON" | jq -r '.[0].id // empty')
  fi
  [ -n "$SELECTED_PTM_ID" ] && echo "export SELECTED_PTM_ID=$SELECTED_PTM_ID" >> .tao_nb_state
fi
echo "SELECTED_PTM_ID=$SELECTED_PTM_ID"


### Train <a class="anchor" id="head-16"></a>

#### View hyperparameters that are enabled for AutoML by default <a class="anchor" id="head-15"></a>

In [ ]:
%%bash
source .tao_nb_state
# View AutoML defaults (if AUTOML_ENABLED=true)
if [ "$AUTOML_ENABLED" = "true" ]; then
  tao $MODEL_NAME get-automl-defaults --output @automl_defaults.json
  cat automl_defaults.json | jq .
fi

#### Set AutoML related configurations <a class="anchor" id="head-16.1"></a>
Refer to these hyper-links to see the parameters supported by each network and add more parameters if necessary in addition to the default automl enabled parameters:

[Deformable Detr](https://github.com/NVIDIA/tao_front_end_services/tree/main/api/specs_utils/specs/deformable_detr/deformable_detr%20-%20train.csv), 
[DINO](https://github.com/NVIDIA/tao_front_end_services/tree/main/api/specs_utils/specs/dino/dino%20-%20train.csv), 

In [ ]:
%%bash
source .tao_nb_state
# View AutoML defaults (if AUTOML_ENABLED=true)

if [ "$AUTOML_ENABLED" = "true" ]; then
  PARAMS=$(cat automl_defaults.json)

  # Algorithm-specific params
  if [ "$AUTOML_ALGORITHM" = "bayesian" ] || [ "$AUTOML_ALGORITHM" = "bfbo" ]; then
    ALGO_PARAMS='{"automl_max_recommendations": 20}'
  elif [ "$AUTOML_ALGORITHM" = "hyperband" ]; then
    ALGO_PARAMS='{"automl_max_epochs": 27, "automl_reduction_factor": 3, "epoch_multiplier": 1}'
  else
    ALGO_PARAMS='{}'
  fi

  jq -n --arg algo "$AUTOML_ALGORITHM" \
        --argjson params "$PARAMS" \
        --argjson algo_params "$ALGO_PARAMS" '{
    automl_enabled: true,
    automl_algorithm: $algo,
    automl_delete_intermediate_ckpt: false,
    override_automl_disabled_params: false,
    automl_hyperparameters: ($params | tostring),
    algorithm_specific_params: $algo_params,
    metric: "kpi"
  }' > automl_settings.json

  cat automl_settings.json
else
  echo "AutoML is disabled - training will use standard approach"
fi


#### Set AutoML related configurations <a class="anchor" id="head-16.1"></a>
Refer to these hyper-links to see the parameters supported by each network and add more parameters if necessary in addition to the default automl enabled parameters: 

[Deformable Detr](https://github.com/NVIDIA/tao_front_end_services/tree/main/api/specs_utils/specs/deformable_detr/deformable_detr%20-%20train.csv), 
[DINO](https://github.com/NVIDIA/tao_front_end_services/tree/main/api/specs_utils/specs/dino/dino%20-%20train.csv),

In [ ]:
%%bash
source .tao_nb_state
# For AutoML add --automl-settings key=value to create-job (repeat as needed)


#### Provide train specs

In [ ]:
%%bash
source .tao_nb_state
BASE_OPT=""
[ -n "$SELECTED_PTM_ID" ] && BASE_OPT="--base-experiment-id $SELECTED_PTM_ID"
tao $MODEL_NAME get-job-schema --action train --defaults-only --output @train_spec.yaml
head -80 train_spec.yaml


In [ ]:
%%bash
source .tao_nb_state
# Optional: edit train_spec.yaml with yq or sed, then copy to train_spec_filled.yaml
cp -f train_spec.yaml train_spec_filled.yaml

# Core training overrides
yq -i '.train.num_gpus = 1' train_spec_filled.yaml
yq -i '.train.num_epochs = 10' train_spec_filled.yaml
yq -i '.train.checkpoint_interval = 10' train_spec_filled.yaml
yq -i '.train.validation_interval = 10' train_spec_filled.yaml

# Model-specific overrides: these models require num_classes = N+1 (background class)
if [[ "$MODEL_NAME" != "grounding_dino" ]]; then
  NUM_CLASSES=$((NUM_CLASSES + 1))
  yq -i ".dataset.num_classes = $NUM_CLASSES" train_spec_filled.yaml
fi

cat train_spec_filled.yaml

#### Run train

In [ ]:
%%bash
source .tao_nb_state

# Docker env vars
ENV_OPT=""
[ -n "$HF_TOKEN" ] && ENV_OPT="--env HF_TOKEN=$HF_TOKEN"

AUTOML_OPT=""
[ -f automl_settings.json ] && AUTOML_OPT="--automl-settings @automl_settings.json"

TRAIN_JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action train --name ${MODEL_NAME}_training_job \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --base-experiment-id "$SELECTED_PTM_ID" \
  --train-dataset-uri "$TRAIN_DATASET_URI" --eval-dataset-uri "$EVAL_DATASET_URI" \
  --specs @train_spec_filled.yaml $AUTOML_OPT $ENV_OPT --output json)
JOB_ID_TRAIN=$(echo "$TRAIN_JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$TRAIN_JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_TRAIN=$JOB_ID_TRAIN"
echo "export JOB_ID_TRAIN=$JOB_ID_TRAIN" >> .tao_nb_state


In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_TRAIN"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')

  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0

  [ -z "$STATUS" ] && echo "Waiting..." && sleep 5 && continue
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done

if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi


In [ ]:
%%bash
source .tao_nb_state
# To stop an AutoML job: stop the monitor cell, then run: tao $MODEL_NAME pause-job --job-id $JOB_ID_TRAIN


In [ ]:
%%bash
source .tao_nb_state
# To resume: tao $MODEL_NAME resume-job --job-id $JOB_ID_TRAIN --specs @train_spec_filled.yaml


### Publish model

#### Edit the method of choosing checkpoint from list of train checkpoint files

In [ ]:
%%bash
source .tao_nb_state
# List checkpoint files from the training job
tao $MODEL_NAME list-job-files --job-id "$JOB_ID_TRAIN" --output json | jq .

# Choose checkpoint method: latest_model, best_model, or from_epoch_number
tao $MODEL_NAME update-job --job-id "$JOB_ID_TRAIN" \
  --checkpoint-choose-method best_model

# # Or if you need a specific epoch:
# tao $MODEL_NAME update-job --job-id "$JOB_ID_TRAIN" \
#   --checkpoint-choose-method from_epoch_number \
#   --checkpoint-epoch-number 10

In [ ]:
%%bash
source .tao_nb_state
# Checkpoint selection is per-job; use get-job-schema / create-job with desired checkpoint config.


#### Push model to private ngc team registry

In [ ]:
%%bash
source .tao_nb_state
# tao $MODEL_NAME publish-model --job-id $JOB_ID_TRAIN --display-name "TAO $MODEL_NAME" --description "Train $MODEL_NAME"

#### Remove model from private ngc team registry

In [ ]:
%%bash
source .tao_nb_state
# tao $MODEL_NAME remove-published-model --job-id $JOB_ID --team <team>


### Evaluate <a class="anchor" id="head-17"></a>

#### Provide evaluate specs

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --action evaluate --defaults-only --output @eval_spec.yaml
cp -f eval_spec.yaml eval_spec_filled.yaml


In [ ]:
%%bash
source .tao_nb_state
# Edit eval_spec_filled.yaml if needed (yq/sed)

if [[ "$MODEL_NAME" == "deformable_detr" || "$MODEL_NAME" == "dino" || "$MODEL_NAME" == "rtdetr" ]]; then
  NUM_CLASSES=$((NUM_CLASSES + 1))
  yq -i ".dataset.num_classes = $NUM_CLASSES" eval_spec_filled.yaml
fi

cat eval_spec_filled.yaml

#### Run evaluate

In [ ]:
%%bash
source .tao_nb_state

# Docker env vars
ENV_OPT=""
[ -n "$HF_TOKEN" ] && ENV_OPT="--env HF_TOKEN=$HF_TOKEN"

EVAL_JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action evaluate --name ${MODEL_NAME}_eval \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_TRAIN" \
  --eval-dataset-uri "$EVAL_DATASET_URI" --specs @eval_spec_filled.yaml $ENV_OPT --output json)
JOB_ID_EVAL=$(echo "$EVAL_JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$EVAL_JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_EVAL=$JOB_ID_EVAL"
echo "export JOB_ID_EVAL=$JOB_ID_EVAL" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_EVAL"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi


### Export <a class="anchor" id="head-19"></a>

#### Provide export specs

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --action export --defaults-only --output @export_spec.yaml
cp -f export_spec.yaml export_spec_filled.yaml


In [ ]:
%%bash
source .tao_nb_state
# Edit export_spec_filled.yaml if needed
# Example: Adjust export_spec_filled.yaml for model-specific settings if needed (Deformable DETR, DINO, RTDETR)
if [[ "$MODEL_NAME" == "deformable_detr" || "$MODEL_NAME" == "dino" || "$MODEL_NAME" == "rtdetr" ]]; then
  # Increase num_classes by 1 for these models
  NUM_CLASSES=$((NUM_CLASSES + 1))
  yq -i ".dataset.num_classes = $NUM_CLASSES" export_spec_filled.yaml
fi

if [[ "$MODEL_NAME" == "rtdetr" ]]; then
  # Set input size for RTDETR based on convention (640x640)
  yq e '.export.input_height = 640' -i export_spec_filled.yaml
  yq e '.export.input_width = 640' -i export_spec_filled.yaml
fi

cat export_spec_filled.yaml

#### Run export

In [ ]:
%%bash
source .tao_nb_state

# Docker env vars
ENV_OPT=""
[ -n "$HF_TOKEN" ] && ENV_OPT="--env HF_TOKEN=$HF_TOKEN"

EXPORT_JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action export --name ${MODEL_NAME}_export \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_TRAIN" \
  --train-dataset-uri "$TRAIN_DATASET_URI" --specs @export_spec_filled.yaml $ENV_OPT --output json)
JOB_ID_EXPORT=$(echo "$EXPORT_JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$EXPORT_JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_EXPORT=$JOB_ID_EXPORT"
echo "export JOB_ID_EXPORT=$JOB_ID_EXPORT" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_EXPORT"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi


### Post-training quantization of the model <a class="anchor" id="head-15"></a>
- Quantization can be carried out on either the trainined model or on the ONNX exported model 

#### Get quantize spec

In [ ]:
%%bash
source .tao_nb_state
# Quantization is only supported for rtdetr
if [ "$MODEL_NAME" = "rtdetr" ]; then
  tao $MODEL_NAME get-job-schema --action quantize --defaults-only --output @quantize_spec.yaml
  cp -f quantize_spec.yaml quantize_on_train_spec_filled.yaml
  cp -f quantize_spec.yaml quantize_on_export_spec_filled.yaml
  cat quantize_spec.yaml
else
  echo "Quantization not applicable for $MODEL_NAME - skipping"
fi

In [ ]:
%%bash
source .tao_nb_state
if [ "$MODEL_NAME" = "rtdetr" ]; then
  NUM_CLASSES=$((NUM_CLASSES + 1))
  yq -i ".dataset.num_classes = $NUM_CLASSES" quantize_on_train_spec_filled.yaml
  yq -i '.quantize.layers = [{"module_name": "*", "weights": {"dtype": "float8_e4m3fn"}, "activations": {"dtype": "float8_e4m3fn"}}]' quantize_on_train_spec_filled.yaml
  cat quantize_on_train_spec_filled.yaml
fi

#### Quantization on the trained model <a class="anchor" id="head-15.1"></a>

In [ ]:
%%bash
source .tao_nb_state
if [ "$MODEL_NAME" = "rtdetr" ]; then
  # Docker env vars
  ENV_OPT=""
  [ -n "$HF_TOKEN" ] && ENV_OPT="--env HF_TOKEN=$HF_TOKEN"
  
  QUANTIZE_TRAIN_JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action quantize \
    --name ${MODEL_NAME}_quantize_on_train \
    --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" \
    --parent-job-id "$JOB_ID_TRAIN" --train-dataset-uri "$TRAIN_DATASET_URI" \
    --calibration-dataset-uri "$TRAIN_DATASET_URI" \
    --specs @quantize_on_train_spec_filled.yaml $ENV_OPT --output json)
  JOB_ID_QUANTIZE_ON_TRAIN=$(echo "$QUANTIZE_TRAIN_JOB_RESPONSE" | jq -r '.id // .result')
  MESSAGE=$(echo "$QUANTIZE_TRAIN_JOB_RESPONSE" | jq -r '._message // empty')
  [ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
  echo "JOB_ID_QUANTIZE_ON_TRAIN=$JOB_ID_QUANTIZE_ON_TRAIN"
  echo "export JOB_ID_QUANTIZE_ON_TRAIN=$JOB_ID_QUANTIZE_ON_TRAIN" >> .tao_nb_state
fi

In [ ]:
%%bash
source .tao_nb_state
if [ "$MODEL_NAME" = "rtdetr" ]; then
  JOB_ID="$JOB_ID_QUANTIZE_ON_TRAIN"
  MAX_WAIT_RETRIES=10
  wait_retries=0
  while true; do
    STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
    if [ -z "$STATUS" ]; then
      wait_retries=$((wait_retries + 1))
      echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
      if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
        echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
        exit 1
      fi
      sleep 5
      continue
    fi
    wait_retries=0
    echo "Status: $STATUS"
    tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
    [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
    sleep 15
  done
  if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi
fi

#### Quantization on the exported model <a class="anchor" id="head-15.2"></a>

In [ ]:
%%bash
source .tao_nb_state
if [ "$MODEL_NAME" = "rtdetr" ]; then
  NUM_CLASSES=$((NUM_CLASSES + 1))
  yq -i ".dataset.num_classes = $NUM_CLASSES" quantize_on_export_spec_filled.yaml
  yq -i '.quantize.backend = "modelopt.onnx"' quantize_on_export_spec_filled.yaml
  yq -i '.quantize.mode = "static_ptq"' quantize_on_export_spec_filled.yaml
  yq -i '.quantize.algorithm = "max"' quantize_on_export_spec_filled.yaml
  yq -i '.quantize.layers = [{"module_name": "*", "weights": {"dtype": "float8_e4m3fn"}}]' quantize_on_export_spec_filled.yaml
  yq -i '.quantize.backend_kwargs = {"log_level": "DEBUG", "calibration_eps": ["cpu", "cuda:0", "trt"]}' quantize_on_export_spec_filled.yaml
  cat quantize_on_export_spec_filled.yaml
fi

In [ ]:
%%bash
source .tao_nb_state
if [ "$MODEL_NAME" = "rtdetr" ]; then
  # Docker env vars
  ENV_OPT=""
  [ -n "$HF_TOKEN" ] && ENV_OPT="--env HF_TOKEN=$HF_TOKEN"

  QUANTIZE_EXPORT_JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action quantize \
    --name ${MODEL_NAME}_quantize_on_export \
    --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" \
    --parent-job-id "$JOB_ID_EXPORT" --train-dataset-uri "$TRAIN_DATASET_URI" \
    --calibration-dataset-uri "$TRAIN_DATASET_URI" \
    --specs @quantize_on_export_spec_filled.yaml $ENV_OPT --output json)
  JOB_ID_QUANTIZE_ON_EXPORT=$(echo "$QUANTIZE_EXPORT_JOB_RESPONSE" | jq -r '.id // .result')
  MESSAGE=$(echo "$QUANTIZE_EXPORT_JOB_RESPONSE" | jq -r '._message // empty')
  [ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
  echo "JOB_ID_QUANTIZE_ON_EXPORT=$JOB_ID_QUANTIZE_ON_EXPORT"
  echo "export JOB_ID_QUANTIZE_ON_EXPORT=$JOB_ID_QUANTIZE_ON_EXPORT" >> .tao_nb_state
fi

In [ ]:
%%bash
source .tao_nb_state
if [ "$MODEL_NAME" = "rtdetr" ]; then
  JOB_ID="$JOB_ID_QUANTIZE_ON_EXPORT"
  MAX_WAIT_RETRIES=10
  wait_retries=0
  while true; do
    STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
    if [ -z "$STATUS" ]; then
      wait_retries=$((wait_retries + 1))
      echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
      if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
        echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
        exit 1
      fi
      sleep 5
      continue
    fi
    wait_retries=0
    echo "Status: $STATUS"
    tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
    [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
    sleep 15
  done
  if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi
fi

### TRT Engine generation using TAO-Deploy <a class="anchor" id="head-20"></a>

#### Provide trt engine generation specs

In [ ]:
%%bash
source .tao_nb_state

tao $MODEL_NAME get-job-schema --action gen_trt_engine --defaults-only --output @gen_trt_engine_spec.yaml
cp -f gen_trt_engine_spec.yaml gen_trt_engine_spec_filled.yaml


In [ ]:
%%bash
source .tao_nb_state
# Edit gen_trt_engine_spec_filled.yaml if needed
if [[ "$MODEL_NAME" == "deformable_detr" || "$MODEL_NAME" == "dino" || "$MODEL_NAME" == "grounding_dino" || "$MODEL_NAME" == "rtdetr" ]]; then
  yq -i '.gen_trt_engine.tensorrt.data_type = "FP16"' gen_trt_engine_spec_filled.yaml
fi
if [[ "$MODEL_NAME" == "deformable_detr" || "$MODEL_NAME" == "dino" ]]; then
  NUM_CLASSES=$((NUM_CLASSES + 1))
  yq -i ".dataset.num_classes = $NUM_CLASSES" gen_trt_engine_spec_filled.yaml
fi
cat gen_trt_engine_spec_filled.yaml

#### Run TRT Engine generation

In [ ]:
%%bash
source .tao_nb_state
if [ "$MODEL_NAME" = "rtdetr" ]; then
  PARENT_JOB_ID="$JOB_ID_QUANTIZE_ON_EXPORT"
else
  PARENT_JOB_ID="$JOB_ID_EXPORT"
fi

# Docker env vars
ENV_OPT=""
[ -n "$HF_TOKEN" ] && ENV_OPT="--env HF_TOKEN=$HF_TOKEN"

TRT_JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action gen_trt_engine --name ${MODEL_NAME}_gen_trt_engine \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$PARENT_JOB_ID" \
  --specs @gen_trt_engine_spec_filled.yaml $ENV_OPT --output json)
JOB_ID_TRT=$(echo "$TRT_JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$TRT_JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_TRT=$JOB_ID_TRT"
echo "export JOB_ID_TRT=$JOB_ID_TRT" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_TRT"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### TAO inference <a class="anchor" id="head-21"></a>

#### Provide TAO inference specs

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --action inference --defaults-only --output @tao_inference_spec.yaml
cp -f tao_inference_spec.yaml tao_inference_spec_filled.yaml

In [ ]:
%%bash
source .tao_nb_state
# Edit tao_inference_spec_filled.yaml if needed
if [[ "$MODEL_NAME" == "deformable_detr" || "$MODEL_NAME" == "dino" || "$MODEL_NAME" == "rtdetr" ]]; then
  NUM_CLASSES=$((NUM_CLASSES + 1))
  yq -i ".dataset.num_classes = $NUM_CLASSES" tao_inference_spec_filled.yaml
elif [ "$MODEL_NAME" = "grounding_dino" ]; then
  yq -i '.dataset.infer_data_sources.captions = ["person"]' tao_inference_spec_filled.yaml
fi
cat tao_inference_spec_filled.yaml

#### Run TAO inference

In [ ]:
%%bash
source .tao_nb_state

# Docker env vars
ENV_OPT=""
[ -n "$HF_TOKEN" ] && ENV_OPT="--env HF_TOKEN=$HF_TOKEN"

TAO_INF_JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action inference --name ${MODEL_NAME}_tao_inference \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_TRAIN" \
  --inference-dataset-uri "$EVAL_DATASET_URI" --specs @tao_inference_spec_filled.yaml $ENV_OPT --output json)
JOB_ID_TAO_INF=$(echo "$TAO_INF_JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$TAO_INF_JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_TAO_INF=$JOB_ID_TAO_INF"
echo "export JOB_ID_TAO_INF=$JOB_ID_TAO_INF" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_TAO_INF"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

### TRT inference <a class="anchor" id="head-22"></a>

#### Provide TRT inference specs

In [ ]:
%%bash
source .tao_nb_state
tao $MODEL_NAME get-job-schema --action inference --defaults-only --output @trt_inference_spec.yaml
cp -f trt_inference_spec.yaml trt_inference_spec_filled.yaml

In [ ]:
%%bash
source .tao_nb_state
# Edit trt_inference_spec_filled.yaml if needed
if [[ "$MODEL_NAME" == "deformable_detr" || "$MODEL_NAME" == "dino" || "$MODEL_NAME" == "rtdetr" ]]; then
  NUM_CLASSES=$((NUM_CLASSES + 1))
  yq -i ".dataset.num_classes = $NUM_CLASSES" trt_inference_spec_filled.yaml
  yq -i '.dataset.batch_size = 1' trt_inference_spec_filled.yaml
elif [ "$MODEL_NAME" = "grounding_dino" ]; then
  yq -i '.dataset.infer_data_sources.captions = ["person"]' trt_inference_spec_filled.yaml
  yq -i '.dataset.batch_size = 1' trt_inference_spec_filled.yaml
fi
cat trt_inference_spec_filled.yaml

#### Run TRT inference

In [ ]:
%%bash
source .tao_nb_state

# Docker env vars
ENV_OPT=""
[ -n "$HF_TOKEN" ] && ENV_OPT="--env HF_TOKEN=$HF_TOKEN"

TRT_INF_JOB_RESPONSE=$(tao $MODEL_NAME create-job --kind experiment --action inference --name ${MODEL_NAME}_trt_inference \
  --encryption-key "$ENCODE_KEY" --workspace-id "$WORKSPACE_ID" --parent-job-id "$JOB_ID_TRT" \
  --inference-dataset-uri "$EVAL_DATASET_URI" --specs @trt_inference_spec_filled.yaml $ENV_OPT --output json)
JOB_ID_TRT_INF=$(echo "$TRT_INF_JOB_RESPONSE" | jq -r '.id // .result')
MESSAGE=$(echo "$TRT_INF_JOB_RESPONSE" | jq -r '._message // empty')
[ -n "$MESSAGE" ] && echo "ℹ️  $MESSAGE"
echo "JOB_ID_TRT_INF=$JOB_ID_TRT_INF"
echo "export JOB_ID_TRT_INF=$JOB_ID_TRT_INF" >> .tao_nb_state

In [ ]:
%%bash
source .tao_nb_state
JOB_ID="$JOB_ID_TRT_INF"
MAX_WAIT_RETRIES=10
wait_retries=0
while true; do
  STATUS=$(tao $MODEL_NAME get-job-metadata --job-id "$JOB_ID" --output json 2>/dev/null | jq -r '.status // empty')
  if [ -z "$STATUS" ]; then
    wait_retries=$((wait_retries + 1))
    echo "Waiting... ($wait_retries/$MAX_WAIT_RETRIES)"
    if [ "$wait_retries" -ge "$MAX_WAIT_RETRIES" ]; then
      echo "ERROR: Could not retrieve job status after $MAX_WAIT_RETRIES attempts"
      exit 1
    fi
    sleep 5
    continue
  fi
  wait_retries=0
  echo "Status: $STATUS"
  tao $MODEL_NAME get-job-logs --job-id "$JOB_ID" 2>/dev/null | tail -3
  [ "$STATUS" = "Done" ] || [ "$STATUS" = "Error" ] && break
  sleep 15
done
if [ "$STATUS" = "Error" ]; then echo "ERROR: Job $JOB_ID failed"; exit 1; fi

In [ ]:
%%bash
source .tao_nb_state
# Optional: tao $MODEL_NAME backup-workspace --workspace-id $WORKSPACE_ID --backup_file_name mongodump.tar.gz


### Delete jobs <a class="anchor" id="head-22"></a>

In [ ]:
%%bash
source .tao_nb_state
for j in $JOB_ID_TRAIN $JOB_ID_EVAL $JOB_ID_TAO_INF $JOB_ID_EXPORT $JOB_ID_QUANTIZE_ON_TRAIN $JOB_ID_QUANTIZE_ON_EXPORT $JOB_ID_TRT $JOB_ID_TRT_INF; do
  [ -n "$j" ] && tao $MODEL_NAME delete-job --job-id "$j" --confirm
done